# 02 — GROQ y resumen de emails

## Objetivo
Resumir correos electrónicos con LLMs (primero Gemini, luego Groq) y practicar lectura/escritura de archivos `.txt`.

## Conceptos
- **Prompt engineering**: instrucciones claras para obtener una sola línea de resumen
- **Groq**: API rápida con modelos como `llama-3.3-70b-versatile`
- **Archivos de texto**: guardar y leer listas con `open()`, `read()`, `readlines()`

## Pasos
1. Lista de emails de ejemplo
2. Resumen con Gemini (`generate_content`)
3. Instalación/uso de Groq (en local: ya está en `requirements.txt`)
4. Resumen con Groq (`chat.completions.create`)
5. Guardar resúmenes en `output/lista-de-resumenes.txt`
6. Leer el archivo de tres formas: línea a línea, `read()` completo, `readlines()`

## Requisitos
- `GEMINI_API_KEY` y `GROQ_API_KEY` en `.env`


In [1]:
import sys
sys.path.insert(0, "..")

from notebooks._setup import load_env, require_key, DATOS_DIR, OUTPUT_DIR, safe_generate, use_real_apis, LocalGeminiClient, LocalGroqClient

load_env()
OUTPUT_DIR.mkdir(exist_ok=True)

require_key("GEMINI_API_KEY")
require_key("GROQ_API_KEY")
if use_real_apis():
    from google import genai
    from groq import Groq
    client = genai.Client()
    client_groq = Groq()
else:
    client = LocalGeminiClient()
    client_groq = LocalGroqClient()

# ***GROQ***

In [2]:
cuerpos_correos = [
    "Hola, te envío el reporte de ventas del trimestre que solicitaste en la reunión. Quedo atento a tus comentarios.",
    "¡Tu suscripción ha sido renuevada con éxito! El cargo de 9.99€ se ha aplicado a tu tarjeta terminada en 4421.",
    "¿Qué tal todo? Hace mucho que no hablamos. Podríamos quedar a tomar un café este jueves si tienes un hueco.",
    "Estimado alumno, le recordamos que el plazo para la entrega del proyecto final finaliza este viernes a medianoche."
]

In [3]:
def resumir_correos_gemini(cuerpos_correos):
    lista_resumenes = []
    for numero, correo in enumerate(cuerpos_correos, start=1):
        prompt = f"""Resume en una frase el siguiente correo electronico:

{correo}"""
        resumen = safe_generate(
            lambda texto: client.models.generate_content(model="gemini-2.5-flash", contents=texto),
            prompt,
        )
        lista_resumenes.append(resumen)
        print(f"Correo {numero}: {resumen}")
    return lista_resumenes

In [4]:
# Dependencias instaladas en .venv (ver requirements.txt)


In [5]:
# GROQ_API_KEY cargada en celda de setup

In [6]:
respuesta_demo = safe_generate(
    lambda texto: client_groq.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": "Responde de forma breve."},
            {"role": "user", "content": texto},
        ],
    ).choices[0].message.content,
    "Que puedes hacer?",
)
print(respuesta_demo)

Puedo responder preguntas, resumir textos, clasificar informacion, analizar datos con Python y guardar resultados en tablas o archivos.


In [7]:
def resumir_correos(cuerpos_correos):
    lista_resumenes = []
    for numero, correo in enumerate(cuerpos_correos, start=1):
        prompt = f"""Resume en una frase el siguiente correo electronico:

{correo}"""
        resumen = safe_generate(
            lambda texto: client_groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": texto}],
            ).choices[0].message.content,
            prompt,
        )
        lista_resumenes.append(resumen)
        print(f"Correo {numero}: {resumen}")
    return lista_resumenes

In [8]:
resumir_correos(cuerpos_correos)

Correo 1: Resumen: se envia un reporte de ventas y se solicitan comentarios.
Correo 2: Resumen: se confirma la renovacion de una suscripcion y el cargo aplicado.
Correo 3: Resumen: se propone retomar el contacto y coordinar un encuentro.
Correo 4: Resumen: se recuerda una fecha limite de entrega academica.


['Resumen: se envia un reporte de ventas y se solicitan comentarios.',
 'Resumen: se confirma la renovacion de una suscripcion y el cargo aplicado.',
 'Resumen: se propone retomar el contacto y coordinar un encuentro.',
 'Resumen: se recuerda una fecha limite de entrega academica.']

In [9]:
lista_resumenes = resumir_correos(cuerpos_correos)

Correo 1: Resumen: se envia un reporte de ventas y se solicitan comentarios.
Correo 2: Resumen: se confirma la renovacion de una suscripcion y el cargo aplicado.
Correo 3: Resumen: se propone retomar el contacto y coordinar un encuentro.
Correo 4: Resumen: se recuerda una fecha limite de entrega academica.


In [10]:
lista_resumenes

['Resumen: se envia un reporte de ventas y se solicitan comentarios.',
 'Resumen: se confirma la renovacion de una suscripcion y el cargo aplicado.',
 'Resumen: se propone retomar el contacto y coordinar un encuentro.',
 'Resumen: se recuerda una fecha limite de entrega academica.']

In [11]:
with open(OUTPUT_DIR / "lista-de-resumenes.txt", "w", encoding="utf-8") as archivo:
    for resumen in lista_resumenes:
        archivo.write(resumen + "\n")

In [12]:
nueva_lista_de_lectura = []

with open(OUTPUT_DIR / "lista-de-resumenes.txt", "r", encoding="utf-8") as archivo:
    for linea in archivo:
        nueva_lista_de_lectura.append(linea.strip())

In [13]:
nueva_lista_de_lectura

['Resumen: se envia un reporte de ventas y se solicitan comentarios.',
 'Resumen: se confirma la renovacion de una suscripcion y el cargo aplicado.',
 'Resumen: se propone retomar el contacto y coordinar un encuentro.',
 'Resumen: se recuerda una fecha limite de entrega academica.']

In [14]:
contenido = ""

with open(OUTPUT_DIR / "lista-de-resumenes.txt", "r", encoding="utf-8") as archivo:
    contenido = archivo.read()

In [15]:
contenido

'Resumen: se envia un reporte de ventas y se solicitan comentarios.\nResumen: se confirma la renovacion de una suscripcion y el cargo aplicado.\nResumen: se propone retomar el contacto y coordinar un encuentro.\nResumen: se recuerda una fecha limite de entrega academica.\n'

In [16]:
nueva_lista_de_lectura_2 = []

with open(OUTPUT_DIR / "lista-de-resumenes.txt", "r", encoding="utf-8") as archivo:
    nueva_lista_de_lectura_2 = archivo.readlines()

In [17]:
nueva_lista_de_lectura_2

['Resumen: se envia un reporte de ventas y se solicitan comentarios.\n',
 'Resumen: se confirma la renovacion de una suscripcion y el cargo aplicado.\n',
 'Resumen: se propone retomar el contacto y coordinar un encuentro.\n',
 'Resumen: se recuerda una fecha limite de entrega academica.\n']